PART 1: 
MAX

In [1]:
# -*- coding: utf-8 -*-
"""
ALOP - HZ Determination and Planetary Density Calculation
Author: M.A.J. van Uden s3793680
MAR 2026

For this model, we assume the following conditions:
1. Planetary Mass-dependency is ignored, so HZ distances are uniform for all planets
2. Stellar luminosity is considered constant, and the star is in the Main Sequence.
3. Atmospheres of planets are "cloudless" and feature no greenhouse gases.

"""

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# HZ Formula Coefficients
hz_coeffs = {
    "recent_venus": (1.768,1.3151e-4,5.8695e-10,-2.8895e-12,3.21174e-16), # Optimistic Inner Limit
    "runaway_greenhouse": (1.1105,1.1921e-4,9.5932e-9,-2.6189e-12,1.3710e-16), # Conservative Inner Limit
    "max_greenhouse": (0.3587,5.8087e-5,1.5393e-9,-8.3547e-13,1.0319e-16), # Conservative Outer Limit
    "early_mars": (0.3246,5.213e-5,4.5245e-10,-1.0223e-12,9.6376e-17), # Optimistic Outer Limit
}
# Density coefficient (g/cm^3)
terrestrial_limit = 3.0
earth_density = 5.514

# LOADING DATA
path_planets = r"C:\Universiteit\Twaars vakken\ALOP\Set 3\filtered_planetary_systems_best1.csv.xls"
path_full = r"C:\Universiteit\Twaars vakken\ALOP\Set 3\datanodig.csv.xls"

db_planets = pd.read_csv(path_planets, low_memory=False)
db_full = pd.read_csv(path_full, low_memory=False, comment="#")

# Checks if I have made any spelling errors
def get_columns(db, colname):
    if colname in db.columns:
        return colname
    return None

# Checks for columns (uerr = Upper Error, lerr = Lower Error)
def define_cols(db):
    colmap = {}
    # Star-Related Data ()
    colmap["host_name"] = get_columns(db,"hostname") # name
    colmap["gaia_dr2_id"] = get_columns(db, "gaia_dr2_id")
    colmap["gaia_dr3_id"] = get_columns(db, "gaia_dr3_id")
    colmap["star_teff"] = get_columns(db,"st_teff") # Effective Temperature (K)
    colmap["star_teff_uerr"] = get_columns(db,"st_tefferr1")
    colmap["star_teff_lerr"] = get_columns(db,"st_tefferr2")
    colmap["star_lum"] = get_columns(db,"st_lum") # Luminosity (log10(L_sol))
    colmap["star_lum_uerr"] = get_columns(db,"st_lumerr1")
    colmap["star_lum_lerr"] = get_columns(db,"st_lumerr2")
    colmap["star_rad"] = get_columns(db,"st_rad") # Radius (R_Sol)
    colmap["star_rad_uerr"] = get_columns(db,"st_raderr1")
    colmap["star_rad_lerr"] = get_columns(db,"st_raderr2")
    colmap["star_mass"] = get_columns(db,"st_mass") # Mass (M_Sol)
    colmap["star_mass_uerr"] = get_columns(db,"st_masserr1")
    colmap["star_mass_lerr"] = get_columns(db,"st_masserr2")
    colmap["star_ra"] = get_columns(db,"rastr") # right assencion star (M_Sol)
    colmap["star_dec"] = get_columns(db,"decstr") # Declination 
    
    # Planet-related columns
    colmap["planet_name"] = get_columns(db,"pl_name") # Name
    colmap["pl_per"] = get_columns(db,"pl_orbper") # Orbital Period (days)
    colmap["pl_per_uerr"] = get_columns(db,"pl_orbpererr1")
    colmap["pl_per_lerr"] = get_columns(db,"pl_orbpererr2")
    colmap["pl_sma"] = get_columns(db,"pl_orbsmax") # Semi-Major Axis (AU)
    colmap["pl_sma_uerr"] = get_columns(db,"pl_orbsmaxerr1")
    colmap["pl_sma_lerr"] = get_columns(db,"pl_orbsmaxerr2")
    colmap["pl_ecc"] = get_columns(db,"pl_orbeccen") # Orbital Eccentricity
    colmap["pl_err_uerr"] = get_columns(db,"pl_orbeccenerr1")
    colmap["pl_err_lerr"] = get_columns(db,"pl_orbeccenerr2")
    colmap["pl_mass"] = get_columns(db,"pl_bmasse") # Mass (in M_Earth)
    colmap["pl_mass_uerr"] = get_columns(db,"pl_bmasseerr1")
    colmap["pl_mass_lerr"] = get_columns(db,"pl_bmasseerr2")
    colmap["pl_rad"] = get_columns(db,"pl_rade") # Radius (in M_Earth)
    colmap["pl_rad_uerr"] = get_columns(db,"pl_radeerr1")
    colmap["pl_rad_lerr"] = get_columns(db,"pl_radeerr2")
    colmap["pl_den"] = get_columns(db,"pl_dens") # Density (in g/cm^3)
    colmap["pl_den_uerr"] = get_columns(db,"pl_denserr1")
    colmap["pl_den_lerr"] = get_columns(db,"pl_denserr2")
    return colmap

colmap = define_cols(db_full)
#print(colmap) # check/debug to ensure all columns are read correctly

# Implementation of the Kopparapu-esque definition of the HZ
# Input: T_eff (K), HZ Coefficients
# Output: S_eff
def calc_seff(teff,limitname):
    if teff is None or pd.isna(teff):
        return None
    S0, a, b, c, d = hz_coeffs[limitname]
    T = teff - 5780.0
    return S0 + a*T + b*(T**2) + c*(T**3) + d*(T**4)

# Implementation of S_eff to AU conversion
# Input: Stellar Luminosity (L_sol), Effective Temperature (K)
# Output: HZ Limits (AU)
def compute_hz_dist(st_lum, teff):
    if pd.isna(teff) or pd.isna(st_lum): # debug if one contains NaNs
        return{k: None for k in ["inner_optimistic","inner_conservative","outer_conservative","outer_optimistic"]}
    L = 10**st_lum # logarithmic converion
    s_venus = calc_seff(teff,"recent_venus")
    s_runaway = calc_seff(teff,"runaway_greenhouse")
    s_maxgh = calc_seff(teff,"max_greenhouse")
    s_mars = calc_seff(teff,"early_mars")
    out = {} # catch dict for output
    out["inner_optimistic"] = np.sqrt(L/s_venus) if (s_venus and s_venus>0) else None
    out["inner_conservative"] = np.sqrt(L/s_runaway) if (s_runaway and s_runaway>0) else None
    out["outer_conservative"] = np.sqrt(L/s_maxgh) if (s_maxgh and s_maxgh>0) else None
    out["outer_optimistic"] = np.sqrt(L/s_mars) if (s_mars and s_mars>0) else None
    return out

# Parent function that will calculate HZ with above functions per star
# Input: Database, column map
# Output: HZ limits (AU) per star
def compute_hz_star(db, colmap):
    hosts = colmap["host_name"]
    teffs = colmap["star_teff"]
    loglums = colmap["star_lum"] # DB has log10(L_star), which needs to be converted (see above)
    
    # Instead of repeating calc for all planets in the same system, we shorten the list to
    # only feature the first occurrence of each host star by building a new dataframe
    hosts_df = db[[hosts]].drop_duplicates().copy()
    hosts_df = hosts_df.rename(columns={hosts:"host_name"})
    hz_list = [] # catch array
    for host in hosts_df["host_name"]:
        subset = db[db[hosts] == host] # picks first row where data is available
        teff = None
        lum = None
        # Cleanup
        if teffs is not None:
            teff = subset[teffs].dropna().iloc[0] if subset[teffs].dropna().shape[0]>0 else None
        if loglums is not None:
            loglum = subset[loglums].dropna().iloc[0] if subset[loglums].dropna().shape[0]>0 else None
        # Passes arrays onto T_eff -> S_Eff -> AU converter functions
        hz = compute_hz_dist(loglum, teff)
        row = {'host_name': host, 'st_teff': teff, 'st_loglum': loglum}
        row.update(hz)
        hz_list.append(row)
    return pd.DataFrame(hz_list)

# Function to check whether planet is (partially) inside the HZ of its host star
# Input: Planet SMA (AU), Eccentricity, HZ Inner and Outer Limits (AU)
#        Orbital Period (days), Stellar Mass (M_Sol)
# Output: inside (fully inside), partial (partially inside), outside (fully outside)
# Note I will use peri/Pe for periapsis and apo/Ap for apoapsis
def planet_hz_check(a, p, e, M, hz_inner, hz_outer):
    if a is None or pd.isna(a): # If SMA is missing, calculate via Kepler's Third Law
        if pd.isna(p) or pd.isna(M): # Not possible if M* and/or P is missing
            return 'unknown'  
        p_yr = p / 365.25 # days to Julian Years conversion
        a = (p_yr**2 * M)**(1/3)
    e = 0.0 if (e is None or pd.isna(e)) else e
    peri = a * (1 - e)
    apo = a * (1 + e)
    if hz_inner is None or hz_outer is None:
        return 'unknown' # If HZ could not be calculated, skip
    if peri >= hz_inner and apo <= hz_outer:
        return 'inside'
    # 3 scenarios:
        # periapsis below inner limit but apoapsis inside
        # periapsis inside but apoapsis above outer limit
        # periapsis below inner limit AND apoapsis above outer limit
    if (peri < hz_inner and apo > hz_inner) or (peri < hz_outer and apo > hz_outer) or (peri < hz_inner and apo > hz_outer):
        return 'partial'
    return 'outside'

# Function to compute the density of a given planet using radius and mass in Earth values
# If the density is already given in the database, we use that value instead.
# Input: Planet Mass (M_Earth), Planet Radius (R_Earth); Planet Density (g/cm^3)
# Output: Planet Density (g/cm^3), Calculation Class (Catalog/Computed/Insufficient Data)
def compute_density(row, colmap):
    dens_col = colmap.get('pl_dens')
    if dens_col and dens_col in row.index and not pd.isna(row.get(dens_col)):
        return row.get(dens_col), 'catalog' # assigns "Catalog" class ==> density found in database
    m_e_col = colmap.get('pl_mass')
    r_e_col = colmap.get('pl_rad')
    # Mass retrieval (assigns None by default, overwrites with found value)
    mass_earth = None
    if m_e_col and m_e_col in row.index and not pd.isna(row.get(m_e_col)):
        mass_earth = row.get(m_e_col)
    # Radius retrieval
    radius_earth = None
    if r_e_col and r_e_col in row.index and not pd.isna(row.get(r_e_col)):
        radius_earth = row.get(r_e_col)
    # if there is one or both values missing, assign "Insufficient Data" class ==> Density could not be calculated.
    if mass_earth is None or radius_earth is None:
        return None, 'insufficient_data'
    density = (mass_earth / (radius_earth**3.0)) * earth_density
    return density, 'computed' # assings "Computed" class ==> Density is manually computed

# Function to determine whether a planet is Gaseous (M>10ME and/or rho<3.0g/cm^3) or Terrestrial
# Input: Planetary Density (g/cm^3), Density Threshold (g/cm^3)
# Output: Planet Class (Terrestrial/Gaseous/Unknown)
# Note: Terrestrial planets only go up to 10 ME, above which a planet defaults to Gaseous
def classify_density_type(masse, density, threshold=terrestrial_limit):
    if masse >= 10.0: # Above 10 Earth masses
       return 'gaseous'
    if density is None or pd.isna(density):
        return 'unknown' # yet another fallback if compute_density was unable to calculate density
    return 'terrestrial' if density >= threshold else 'gaseous'

# ----------------------------------- ANALYSIS OF THE DATA --------------------------------------

# Activates the procedure above
hz_df = compute_hz_star(db_planets, colmap)

# Merges HZ limits into the planets table, matching on host star name.
host_col = colmap['host_name']
if host_col is None: # debug
    raise RuntimeError("Could not detect host column name.")
planets_with_hz = db_planets.merge(hz_df, left_on=host_col, right_on='host_name', how='left')

# "r" as input in the following functions denote the data rows.

# Function to activate the HZ check and applies it to the data
def classify_pl_hz(r):
    a = r.get(colmap['pl_sma'])  # semi-major axis (AU)
    p = r.get(colmap['pl_per'])  # period (days)
    e = r.get(colmap['pl_ecc'])  # eccentricity
    M = r.get(colmap['star_mass']) # Stellar Mass (M_Sun)
    # Optimistic HZ limits (Recent-Venus to Early-Mars)
    inner = r.get('inner_optimistic') 
    outer = r.get('outer_optimistic')
    return planet_hz_check(a, p, e, M, inner, outer)

planets_with_hz['hz_membership'] = planets_with_hz.apply(classify_pl_hz, axis=1)

# Function to compute densities and classify them based on density
def add_density_info(r):
    m = r.get(colmap['pl_mass'])
    dens, method = compute_density(r, colmap)
    ptype = classify_density_type(m, dens)
    # Returns a pandas Series w/ density, Calculation Class, Planet Classification
    return pd.Series({'density_gcm3': dens, 'density_method': method, 'planet_type': ptype})

# And adds said series to the database.
planets_with_hz[['density_gcm3','density_method','planet_type']] = planets_with_hz.apply(add_density_info, axis=1)

# FUNCTION FOR SOLAR-LIKENESS CHECK
# This function checks whether the planetary system fulfills the three conditions:
    # 25-50% of planets in HZ
    # 35-65% of planets are terrestrial
    # All Terrestral planets orbit closer than the Gaseous planets
# Input: Planetary Systems dataframe
# Output: Summary of planetary systems.
def system_summary(df_system):
    n = len(df_system)
    n_hz = (df_system['hz_membership'].isin(['inside','centered','partial'])).sum()
    frac_hz = n_hz / n if n>0 else 0.0
    n_terrestrial = (df_system['planet_type'] == 'terrestrial').sum()
    frac_ter = n_terrestrial / n if n>0 else 0.0

    # Architecture check: SMA of closest gas planet is larger than SMA of most distant terrestrial planet
    # Retrieve SMA data and drop any None values; auto-assign False value (not ordered)
    ter_smas = df_system[df_system['planet_type']=='terrestrial'][colmap['pl_sma']].dropna()
    gas_smas = df_system[df_system['planet_type']=='gaseous'][colmap['pl_sma']].dropna()
    architecture_ok = False
    if len(ter_smas)>0 and len(gas_smas)>0: # Only if there are planets of both classes, check:
        architecture_ok = ter_smas.max() < gas_smas.min() # Returns True if ordered
    
    # Final Solar-Like check (True if all conditions are met)
    solar_like = (0.35 <= frac_ter <= 0.65) and architecture_ok and (0.25 <= frac_hz <= 0.50) #and architecture_ok  
    return {
        'n_planets': n, 'frac_hz': frac_hz, 'frac_ter': frac_ter,
        'architecture_ok': architecture_ok, 'solar_like': solar_like
    }

# Groups planets together per planetary system
grouped = planets_with_hz.groupby('host_name')
summary_rows = [] # catch array
for host, group in grouped: 
    s = system_summary(group) # executes system summary per system
    s['host_name'] = host # adds name of host star
    summary_rows.append(s) # and adds it to the catch

# creates a new dataframe from the summary catch and selects Solar-like systems
summary_df = pd.DataFrame(summary_rows) 
solar_like_systems = summary_df[summary_df['solar_like']] 
print("Solar-like candidate hosts:")
print(solar_like_systems)

# Checks for missing values in each of the columns in the dataframes/-base
def check_missing_values(df, colmap):
    results = []

    for key, col in colmap.items():
        if col is None or col not in df.columns:
            results.append({
                'parameter': key,
                'column': col,
                'missing_count': 'COLUMN NOT FOUND',
                'missing_fraction': None
            })
            continue

        missing = df[col].isna().sum()
        total = len(df)

        results.append({
            'parameter': key,
            'column': col,
            'missing_count': missing,
            'missing_fraction': missing / total
        })

    return pd.DataFrame(results)
missing_df = check_missing_values(db_planets, colmap)
print(missing_df)

# Inspection function to list all data from one planetary system
# Input: databases of HZ, column map, target host name
# Output: literally everything calculated above
def inspect_system(df, colmap, hz_df, host_name):
    host_col = colmap['host_name']

    if host_col is None:
        print("Host column not found!")
        return

    system = df[df[host_col] == host_name]

    if system.empty:
        print(f"No system found for {host_name}")
        return

    print(f"\n===== SYSTEM: {host_name} =====")

# Stellar properties
    print("\n--- Stellar Properties ---")

    def get_first_valid(system, col):
        if col is None or col not in system.columns:
            return None
        
        vals = system[col]
        
        # Drop NaNs
        vals = vals[~pd.isna(vals)]
        
        if len(vals) == 0:
            return None

        return vals.iloc[0]

    teff = get_first_valid(system, colmap['star_teff'])
    lum_log = get_first_valid(system, colmap['star_lum'])
    radius = get_first_valid(system,colmap['star_rad'])
    mass = get_first_valid(system, colmap['star_mass'])


    # Convert luminosity (NASA db has luminosity in log10(L_Sol))
    lum = 10**lum_log if lum_log is not None and not pd.isna(lum_log) else None

    print(f"T_eff: {teff}")
    print(f"Luminosity log10(L/Lsun): {lum_log}")
    print(f"Luminosity (L/Lsun): {lum}")
    print(f"Radius (Rsun): {radius}")
    print(f"Mass (Msun): {mass}")

# HZ Limits
    print("\n--- Habitable Zone Limits (AU) ---")

    hz_row = hz_df[hz_df['host_name'] == host_name]

    if not hz_row.empty:
        hz_row = hz_row.iloc[0]

        print(f"Inner (Optimistic):     {hz_row['inner_optimistic']}")
        print(f"Inner (Conservative):   {hz_row['inner_conservative']}")
        print(f"Outer (Conservative):   {hz_row['outer_conservative']}")
        print(f"Outer (Optimistic):     {hz_row['outer_optimistic']}")

        hz_inner = hz_row['inner_optimistic']
        hz_outer = hz_row['outer_optimistic']
    else:
        print("No HZ data available.")
        hz_inner = hz_outer = None

# Table containing all data of the planets
    print("\n--- Planets ---")

    rows = []

    for _, row in system.iterrows():
        name = row.get(colmap['planet_name'])

        a = row.get(colmap['pl_sma'])
        P = row.get(colmap['pl_per'])
        e = row.get(colmap['pl_ecc'])
        M = row.get(colmap["star_mass"])
        m = row.get(colmap["pl_mass"])

        # Fix eccentricity
        if pd.isna(e):
            e = 0.0
        
        # 3rd Law Kepler calc for inspection
        # Note: this is only to show the SMA in the inspection
        # It is already calced in the hz check func
        def compute_sma_from_period(P_days, M_star):
            if pd.isna(P_days) or pd.isna(M_star):
                return None

            P_years = P_days / 365.25
            return (P_years**2 * M_star) ** (1/3)
        
        if pd.isna(a):
            a = compute_sma_from_period(P, mass)

# HZ check
        hz_status = planet_hz_check(a, P, e,M, hz_inner, hz_outer)

# Density calcs
        density, method = compute_density(row, colmap)
        ptype = classify_density_type(m, density)

        rows.append({
            "planet": name,
            "SMA (AU)": a,
            "Period (d)": P,
            "Eccentricity": e,
            "HZ status": hz_status,
            "Density": density,
            "Type": ptype,
            "Density method": method
        })

    result_df = pd.DataFrame(rows)

    # Sort by SMA for readability
    result_df = result_df.sort_values(by="SMA (AU)")

    print(result_df.to_string(index=False))
# Insert target system here as string
inspect_system(planets_with_hz, colmap, hz_df, "Kepler-139")



Solar-like candidate hosts:
    n_planets  frac_hz  frac_ter  architecture_ok  solar_like  host_name
6           4     0.25       0.5             True        True     GJ 887
86          4     0.25       0.5             True        True  Kepler-68
         parameter           column  missing_count  missing_fraction
0        host_name         hostname              0          0.000000
1      gaia_dr2_id      gaia_dr2_id              0          0.000000
2      gaia_dr3_id      gaia_dr3_id              4          0.008421
3        star_teff          st_teff              0          0.000000
4   star_teff_uerr      st_tefferr1              7          0.014737
5   star_teff_lerr      st_tefferr2              7          0.014737
6         star_lum           st_lum              0          0.000000
7    star_lum_uerr       st_lumerr1            165          0.347368
8    star_lum_lerr       st_lumerr2            165          0.347368
9         star_rad           st_rad              0          0.0

Addition from Sim to distinguish between Solarlikeniss

In [2]:
def system_summary(df_system, conditions):
    n = len(df_system)
    n_hz = (df_system['hz_membership'].isin(['inside','centered','partial'])).sum()
    frac_hz = n_hz / n if n>0 else 0.0
    n_terrestrial = (df_system['planet_type'] == 'terrestrial').sum()
    frac_ter = n_terrestrial / n if n>0 else 0.0

    # Architecture check: SMA of closest gas planet is larger than SMA of most distant terrestrial planet
    # Retrieve SMA data and drop any None values; auto-assign False value (not ordered)
    ter_smas = df_system[df_system['planet_type']=='terrestrial'][colmap['pl_sma']].dropna()
    gas_smas = df_system[df_system['planet_type']=='gaseous'][colmap['pl_sma']].dropna()
    architecture_ok = False
    if len(ter_smas)>0 and len(gas_smas)>0: # Only if there are planets of both classes, check:
        architecture_ok = ter_smas.max() < gas_smas.min() # Returns True if ordered
    
    # Different kinds of conditions to test
    full = (0.35 <= frac_ter <= 0.65) and (0.25 <= frac_hz <= 0.50) and architecture_ok

    quasi_1 = (0.35 <= frac_ter <= 0.65) and (0.25 <= frac_hz <= 0.50)
    quasi_2 = (0.25 <= frac_hz <= 0.50) and architecture_ok
    quasi_3 = (0.35 <= frac_ter <= 0.65) and architecture_ok

    semi_1 = (0.35 <= frac_ter <= 0.65)
    semi_2 = (0.25 <= frac_hz <= 0.50)
    semi_3 = architecture_ok

    all_conditions = [full, quasi_1, quasi_2, quasi_3, semi_1, semi_2, semi_3]
    # Final Solar-Like check (True if all conditions are met)
    solar_like = (0.35 <= frac_ter <= 0.65) and (0.25 <= frac_hz <= 0.50) #and architecture_ok  
    solar_like = all_conditions[conditions] 
    return {
        'n_planets': n, 'frac_hz': frac_hz, 'frac_ter': frac_ter,
        'architecture_ok': architecture_ok, 'solar_like': solar_like
    }

def group_stuff(conditions):
    # Groups planets together per planetary system
    grouped = planets_with_hz.groupby('host_name')
    summary_rows = [] # catch array
    for host, group in grouped: 
        s = system_summary(group, conditions) # executes system summary per system
        s['host_name'] = host # adds name of host star
        summary_rows.append(s) # and adds it to the catch

    # creates a new dataframe from the summary catch and selects Solar-like systems
    summary_df = pd.DataFrame(summary_rows) 
    return summary_df[summary_df['solar_like']] 


#grouping stuff for different conditions
full_solar_like = group_stuff(0)

quasi_solar_like_1 = group_stuff(1)
quasi_solar_like_2 = group_stuff(2)
quasi_solar_like_3 = group_stuff(3)

semi_solar_like_1 = group_stuff(4)
semi_solar_like_2 = group_stuff(5)
semi_solar_like_3 = group_stuff(5)

all_full = full_solar_like

print(all_full)

combined_quasi = pd.concat([quasi_solar_like_1, quasi_solar_like_2, quasi_solar_like_3]).drop_duplicates()
combined_semi = pd.concat([semi_solar_like_1, semi_solar_like_2, semi_solar_like_3]).drop_duplicates()

#print(combined_quasi['host_name'], len(combined_quasi))
#print(combined_semi['host_name'], len(combined_semi))

#determining only solar-like, only quasi-solar-like, only semi-solar-like systems by removing overlaps
only_full = full_solar_like

only_quasi = combined_quasi[~combined_quasi['host_name'].isin(full_solar_like['host_name'])]



only_semi = combined_semi[ ~combined_semi['host_name'].isin(full_solar_like['host_name']) &
                          ~combined_semi['host_name'].isin(combined_quasi['host_name'])]

print(only_quasi, len(only_quasi))
print(only_semi, len(only_semi))



    n_planets  frac_hz  frac_ter  architecture_ok  solar_like  host_name
6           4     0.25       0.5             True        True     GJ 887
86          4     0.25       0.5             True        True  Kepler-68
    n_planets  frac_hz  frac_ter  architecture_ok  solar_like   host_name
3           4      0.5     0.250             True        True     GJ 3293
13          4      0.5     0.750             True        True    HD 20794
2           4      0.0     0.500             True        True      DMPP-1
14          6      0.0     0.500             True        True   HD 219134
16          4      0.0     0.500             True        True     HD 3167
25          8      0.0     0.375             True        True     KOI-351
36          5      0.0     0.400             True        True  Kepler-154
40          5      0.0     0.600             True        True  Kepler-169
62          4      0.0     0.500             True        True  Kepler-282
65          5      0.0     0.400         

PART 2:
Femke

In [3]:
"""
Making a Gaia Data set 
Femke
APR 2026
"""
%pip install astroquery
from astropy.coordinates import SkyCoord
import astropy.units as u

def parse_coords(ra_str, dec_str):
    try:
        c = SkyCoord(ra_str + " " + dec_str, unit=(u.hourangle, u.deg))
        return c.ra.deg, c.dec.deg
    except:
        return None, None

db_planets[['ra_deg','dec_deg']] = db_planets.apply(
    lambda r: pd.Series(parse_coords(r[colmap['star_ra']], r[colmap['star_dec']])),
    axis=1
)


from astroquery.gaia import Gaia
def query_gaia(ra, dec, radius=3.5):  # arcsec
    if ra is None or dec is None:
        return None

    query = f"""
    SELECT TOP 1
        gs.source_id,
        gs.parallax,
        gs.parallax_error,
        gs.phot_g_mean_mag,
        gs.phot_bp_mean_mag, 
        gs.phot_rp_mean_mag,
        gs.ruwe,

        ap.teff_gspphot,
        ap.teff_gspphot_lower,
        ap.teff_gspphot_upper,

        ap.logg_gspphot,
        ap.mh_gspphot,

        ap.distance_gspphot

    FROM gaiadr3.gaia_source AS gs
    JOIN gaiadr3.astrophysical_parameters AS ap
    ON gs.source_id = ap.source_id

    WHERE CONTAINS(
        POINT('ICRS', gs.ra, gs.dec),
        CIRCLE('ICRS', {ra}, {dec}, {radius/3600.})
    ) = 1

    ORDER BY gs.phot_g_mean_mag ASC
    """

    job = Gaia.launch_job(query)
    results = job.get_results()

    return results[0] if len(results) > 0 else None
gaia_rows = []

# Only keep relevant hosts
selected_hosts = solar_like_systems['host_name']

# Reduce dataset to only those hosts (keeps RA/Dec)
subset = db_planets[db_planets[colmap['host_name']].isin(selected_hosts)]

# Drop duplicates → one query per star
subset = subset[[colmap['host_name'], 'ra_deg', 'dec_deg']].drop_duplicates()

import time

print(subset)
print(f"Looking up {len(subset)} stars in Gaia archive")

for _, row in subset.iterrows():
    ra = row['ra_deg']
    dec = row['dec_deg']
    host = row[colmap['host_name']]

    res = None
    max_pogingen = 3 # How often we try again if it crashes
    
    for poging in range(max_pogingen):
        try:
            res = query_gaia(ra, dec)
            break # If we have successfully found the star, it doesn't need to be attempted again
        except Exception as e:
            print(f"Can't find {host} (Poging {poging+1}/{max_pogingen}) so we will now wait 3 seconds")
            time.sleep(3) # wait for 3 secs until another attempt is made

    if res is None:
        print(f"-> Definitief overgeslagen: {host} (server bleef weigeren).")
        continue # if it still can't find the star, we move on. it is what it is

    def g(key):
        return res[key] if key in res.colnames else None

    gaia_rows.append({
        'host_name': host,
        'source_id': g('source_id'),
        'parallax': g('parallax'),
        'parallax_error': g('parallax_error'),
        'phot_g': g('phot_g_mean_mag'),
        'phot_bp' : g('phot_bp_mean_mag'),
        'phot_rp' : g('phot_rp_mean_mag'),
        'teff': g('teff_gspphot'),
        'teff_lower': g('teff_gspphot_lower'),
        'teff_upper': g('teff_gspphot_upper'),
        'logg': g('logg_gspphot'),
        'feh': g('mh_gspphot'),
        'distance': g('distance_gspphot'),
        'ruwe': g('ruwe'),
        'mass_flame': g('mass_flame'),
        'age_flame':  g('age_flame'),
        'age_flame_lower': g('age_flame_lower'),
        'age_flame_upper': g('age_flame_upper'),
    })
    
    # standard waiting time in between successfull queries
    time.sleep(1.5) 

gaia_df = pd.DataFrame(gaia_rows)


print(gaia_df)
print(len(gaia_df))

#%%


Note: you may need to restart the kernel to use updated packages.
In preparation for Gaia DR4, the Gaia archive is in evolution. Unfortunately, it may be unstable at times and particular types of queries may time out. Please consider registering for a user account (https://www.cosmos.esa.int/web/gaia-users/register). For questions or advice, please contact the Gaia helpdesk (https://www.cosmos.esa.int/web/gaia/gaia-helpdesk).
      hostname      ra_deg    dec_deg
25      GJ 887  346.502750 -35.847350
377  Kepler-68  291.032292  49.040214
Looking up 2 stars in Gaia archive
   host_name            source_id    parallax  parallax_error    phot_g  \
0     GJ 887  6553614253923452800  304.135369        0.019996  6.522032   
1  Kepler-68  2129550445852902656    6.929800        0.009994  9.938094   

     phot_bp   phot_rp         teff   teff_lower   teff_upper    logg     feh  \
0   7.589791  5.491506  3376.084473  3366.029785  3392.150635  4.1558 -0.9754   
1  10.256315  9.454888  5729.4624

In [4]:
print(only_quasi['host_name'])

3        GJ 3293
13      HD 20794
2         DMPP-1
14     HD 219134
16       HD 3167
25       KOI-351
36    Kepler-154
40    Kepler-169
62    Kepler-282
65    Kepler-292
83     Kepler-55
87    Kepler-758
Name: host_name, dtype: object


Addition from Sim for similar purposes as before (solarlikeness)


In [ ]:
#make it a function so we can reuse it for the other conditions as well
def find_stars(systems):

    from astropy.coordinates import SkyCoord
    import astropy.units as u

    def parse_coords(ra_str, dec_str):
        try:
            c = SkyCoord(ra_str + " " + dec_str, unit=(u.hourangle, u.deg))
            return c.ra.deg, c.dec.deg
        except:
            return None, None

    db_planets[['ra_deg','dec_deg']] = db_planets.apply(
        lambda r: pd.Series(parse_coords(r[colmap['star_ra']], r[colmap['star_dec']])),
        axis=1
    )

    from astroquery.gaia import Gaia
    def query_gaia(ra, dec, radius=15):  # arcsec, #increased radius relative to before to account for potential mismatches in coordinates and to increase the chance of finding the star in Gaia
        if ra is None or dec is None:
            return None

        #LEFTJOIN instead of JOIN suggested by claude to fix nt finding stars
        query = f"""
        SELECT TOP 1
            gs.source_id,
            gs.parallax,
            gs.parallax_error,
            gs.phot_g_mean_mag,
            gs.phot_bp_mean_mag, 
            gs.phot_rp_mean_mag,
            gs.ruwe,

            ap.teff_gspphot,
            ap.teff_gspphot_lower,
            ap.teff_gspphot_upper,

            ap.mass_flame,
            ap.age_flame,
            ap.age_flame_lower,
            ap.age_flame_upper,

            ap.logg_gspphot,
            ap.mh_gspphot,

            ap.distance_gspphot

        FROM gaiadr3.gaia_source AS gs
        LEFT JOIN gaiadr3.astrophysical_parameters AS ap 
        ON gs.source_id = ap.source_id

        WHERE CONTAINS(
            POINT('ICRS', gs.ra, gs.dec),
            CIRCLE('ICRS', {ra}, {dec}, {radius/3600.})
        ) = 1

        ORDER BY gs.phot_g_mean_mag ASC
        """

        job = Gaia.launch_job(query)
        results = job.get_results()

        return results[0] if len(results) > 0 else None
    gaia_rows = []

    # Only keep relevant hosts
    selected_hosts = systems

    # Reduce dataset to only those hosts (keeps RA/Dec)
    subset = db_planets[db_planets[colmap['host_name']].isin(selected_hosts)]

    # Drop duplicates → one query per star
    subset = subset[[colmap['host_name'], 'ra_deg', 'dec_deg']].drop_duplicates()

    import time

    print(subset)
    print(f"Looking up {len(subset)} stars in Gaia archive")

    for _, row in subset.iterrows():
        ra = row['ra_deg']
        dec = row['dec_deg']
        host = row[colmap['host_name']]

        res = None
        max_pogingen = 3 # How often we try again if it crashes
        
        for poging in range(max_pogingen):
            try:
                res = query_gaia(ra, dec)
                break # If we have successfully found the star, it doesn't need to be attempted again
            except Exception as e:
                print(f"Can't find {host} (Poging {poging+1}/{max_pogingen}) so we will now wait 3 seconds")
                time.sleep(5) # wait for 3 secs until another attempt is made

        if res is None:
            print(f"-> Definitief overgeslagen: {host} (server bleef weigeren).")
            continue # if it still can't find the star, we move on. it is what it is

        def g(key):
            return res[key] if key in res.colnames else None

        gaia_rows.append({
            'host_name': host,
            'source_id': g('source_id'),
            'parallax': g('parallax'),
            'parallax_error': g('parallax_error'),
            'phot_g': g('phot_g_mean_mag'),
            'phot_bp' : g('phot_bp_mean_mag'),
            'phot_rp' : g('phot_rp_mean_mag'),
            'teff': g('teff_gspphot'),
            'teff_lower': g('teff_gspphot_lower'),
            'teff_upper': g('teff_gspphot_upper'),
            'logg': g('logg_gspphot'),
            'feh': g('mh_gspphot'),
            'distance': g('distance_gspphot'),
            'ruwe': g('ruwe'),
            'mass': g('mass_flame'),
            'age':  g('age_flame'),
            'age_lower': g('age_flame_lower'),
            'age_upper': g('age_flame_upper'),
            'ra': ra,
            'dec': dec,
        })
        
        # standard waiting time in between successfull queries
        time.sleep(5.0) 

    gaia_df = pd.DataFrame(gaia_rows)
    return gaia_df

"""print(solar_like_systems['host_name'], only_full['host_name'])

stars = find_stars(solar_like_systems['host_name'])
print(stars)
print(len(stars))"""

#applyinf function to different conditions
full_stars = find_stars(full_solar_like['host_name'])
print(full_stars)

quasi_stars = find_stars(only_quasi['host_name'])
print(quasi_stars)

semi_stars = find_stars(only_semi['host_name'])
print(semi_stars)

      hostname      ra_deg    dec_deg
25      GJ 887  346.502750 -35.847350
377  Kepler-68  291.032292  49.040214
Looking up 2 stars in Gaia archive


PART 3:
Benthe

In [ ]:
"""
Determining the spectral types and stellar ages using isochrones
Benthe
April 2026
"""
# First, we determine the stellar type based on the effective temperature of the star.
def find_spectral_type(teff):
    
    # not needed because the dataset already contains the stellar types
    
    if teff >= 29200:
        return "O"
    elif teff >= 9600:
        return "B"
    elif teff >= 7350:
        return "A"
    elif teff >= 6050:
       return "F"
    elif teff >= 5240:
        return "G"
    elif teff >= 3750:
        return "K"
    else:
        return "M"

# Applying this function to the Gaia data
gaia_df["spectral_type"] = gaia_df["teff"].apply(find_spectral_type)

In [ ]:


print(len(gaia_df))

from astroquery.vizier import Vizier

valid_gaia_ids = gaia_df['source_id'].dropna().astype('int64').tolist()

# Select only the rows with valid Gaia IDs
valid_gaia_df = gaia_df[gaia_df['source_id'].isin(valid_gaia_ids)]

print(f"Number of valid Solar-like Gaia IDs to check: {len(valid_gaia_ids)}")

# if no valid star IDs are found, we stop the process
if len(valid_gaia_ids) == 0:
    print("No valid Gaia IDs found to query.")
else:
    
    # initialize Vizier with no row limit (-1)
    v = Vizier(row_limit = -1)

    # We build a query string for multiple Gaia Source IDs
    id_constraints = [str(gid) for gid in valid_gaia_ids]
    id_string = "==" + " | ==".join(id_constraints)

    # We want to check the catalog to see if our found stars are part of a cluster
    try:
        result_clusters = v.query_constraints(
            catalog="J/A+A/673/A114/members", 
            Source=id_string
        )
        
        # in case of no matches found, we print that there are no matches
        if not result_clusters or len(result_clusters[0]) == 0:
            print("None of the solar-like stars are found in a cluster in this catalog.")
            
        # But maybe there are stars that are part of a cluster. If that's the case, we add them to a table
        else:
            member_table = result_clusters[0]
            
            # Extract unique cluster names from the results
            found_clusters = list(set(member_table["Cluster"].tolist()))
            print(f"Match found! Your star(s) belong to the following cluster(s): {found_clusters}")

            print("\nStep 2: Retrieving all members of these identified clusters...")
                
            # We want to find the cluster names associated with the stars
            cluster_string = "==" + " | ==".join(found_clusters)
                
            # Query again to retrieve all stars belonging to these clusters
            all_members_result = v.query_constraints(
                catalog="J/A+A/673/A114/members", 
                Cluster=cluster_string
            )
                
            # If results are found, convert to pandas DataFrame and save to CSV
            if all_members_result:
                cluster_members_df = all_members_result[0].to_pandas()
                output_filename = "solar_like_cluster_members.csv"
                cluster_members_df.to_csv(output_filename, index=False)
                print(f"Success! {len(cluster_members_df)} stars saved to '{output_filename}'.")
            else:
                print("Could not retrieve the members of these clusters.")
                    
    except Exception as e:
        # Catch and print any errors during the query
        print(f"An error occurred during the Vizier query: {e}")
        
""" We now want to find out if the stars that we found are main sequence stars. To do this,
we make a HR diagram, plotting temperature vs absolute magnitude"""

# We'll compute one colour-magnitude diagram that contains all stars
# that were found using the steps above.

# First we compute the stellar colour
gaia_df["bp_rp"] = gaia_df["phot_bp"] - gaia_df["phot_rp"]

# Gaia gives the the apparent magnitude, but since we are interested in
# age, an intrinsic characteristic, we need absolute magnitude.

# We calculate the distance to each star in parsec. The factor 1000 comes
# from the fact that parallax is documented in mas, but the formula needs arcseconds.
"""
gaia_df["distance_parsec"] = 1000 / gaia_df["parallax"]


# Making sure the distance is numeric to avoid errors
gaia_df["distance_parsec"] = pd.to_numeric(gaia_df["distance_parsec"], errors="coerce")

# Initialize column
gaia_df["abs_mag"] = np.nan

# Safer validity mask
valid = ((gaia_df["distance_parsec"] > 0) & gaia_df["distance_parsec"].notna() & gaia_df["phot_g"].notna())

print(gaia_df["distance_parsec"])

gaia_df["distance_parsec"] = pd.to_numeric(gaia_df["distance_parsec"])
"""
# Ensure parallax is clean numeric scalar
gaia_df["parallax"] = gaia_df["parallax"].apply(
    lambda x: float(x) if x is not None and not pd.isna(x) else np.nan
)

# Compute distance (parallax is given in mas, but we need arcsecs to calculate distance)
gaia_df["distance_parsec"] = 1000.0 / gaia_df["parallax"]

# make sure the distance is numeric to avoid errors in the logarithm
gaia_df["distance_parsec"] = pd.to_numeric(gaia_df["distance_parsec"], errors = "coerce")

# validity masks sot he function only takes on valid numbers to avoid errors
valid = ((gaia_df["distance_parsec"] > 0) & gaia_df["distance_parsec"].notna() & gaia_df["phot_g"].notna() & gaia_df["distance_parsec"].notna())

# Compute absolute magnitude
gaia_df.loc[valid, "abs_mag"] = (gaia_df.loc[valid, "phot_g"] + 5 - 5 * np.log10(gaia_df.loc[valid, "distance_parsec"]))


# Printing all found stars' source id, temperature, spectral type and metallicity
# More characteristics can be added easily, by just adding "[characteristic]" to the following line
print(valid_gaia_df[["source_id", "host_name", "teff", "spectral_type", "feh", "distance_parsec"]])



# Plotting the HR diagram
plt.figure(figsize = (8, 8))

# pretty colour gradient
sc = plt.scatter(gaia_df["teff"], gaia_df["abs_mag"], c = gaia_df["bp_rp"], cmap = "coolwarm", s = 20)

#plt.colorbar(sc, label=" Absolute magnitude ") 

# Plotting the sun for reference
plt.plot(5772, 4.83, 'go--', linewidth = 2, marker = "*", markersize = 6,  label = "Sun", color = "lightgreen")
plt.text(5760, 4.92, " Sun", fontsize = 10)

plt.gca().invert_xaxis()
plt.gca().invert_yaxis()

plt.xlabel("Temperature (K)")
plt.ylabel("Absolute magnitude")
plt.title("HR diagram")

plt.show()

# print(gaia_df["distance_parsec"])

PART 4:
Sim


Making Benthe's part a function so we can apply to different data collections

In [ ]:
def find_charachteristics(data):
    data["spectral_type"] = data["teff"].apply(find_spectral_type)
    print(len(data))

    from astroquery.vizier import Vizier

    valid_gaia_ids = data['source_id'].dropna().astype('int64').tolist()
    valid_data = data[data['source_id'].isin(valid_gaia_ids)]
    print(f"Number of valid Solar-like Gaia IDs to check: {len(valid_gaia_ids)}")

    if len(valid_gaia_ids) == 0:
        print("No valid Gaia IDs found to query.")
    else:
        v = Vizier(row_limit=-1)
        id_constraints = [str(gid) for gid in valid_gaia_ids]
        id_string = "==" + " | ==".join(id_constraints)

        try:
            result_clusters = v.query_constraints(
                catalog="J/A+A/673/A114/members",
                Source=id_string
            )
            if not result_clusters or len(result_clusters[0]) == 0:
                print("None of the solar-like stars are found in a cluster in this catalog.")
            else:
                member_table = result_clusters[0]
                found_clusters = list(set(member_table["Cluster"].tolist()))
                print(f"Match found! Your star(s) belong to the following cluster(s): {found_clusters}")
                print("\nStep 2: Retrieving all members of these identified clusters...")
                cluster_string = "==" + " | ==".join(found_clusters)
                all_members_result = v.query_constraints(
                    catalog="J/A+A/673/A114/members",
                    Cluster=cluster_string
                )
                if all_members_result:
                    cluster_members_df = all_members_result[0].to_pandas()
                    output_filename = "solar_like_cluster_members.csv"
                    cluster_members_df.to_csv(output_filename, index=False)
                    print(f"Success! {len(cluster_members_df)} stars saved to '{output_filename}'.")
                else:
                    print("Could not retrieve the members of these clusters.")
        except Exception as e:
            print(f"An error occurred during the Vizier query: {e}")

    # Compute colour
    data["bp_rp"] = data["phot_bp"] - data["phot_rp"]

    # Clean parallax
    data["parallax"] = data["parallax"].apply(
        lambda x: float(x) if x is not None and not pd.isna(x) else np.nan
    )

    # Compute distance in parsec (parallax in mas → divide by 1000 for arcsec)
    data["distance_parsec"] = 1000.0 / data["parallax"]
    data["distance_parsec"] = pd.to_numeric(data["distance_parsec"], errors="coerce")

    # Validity mask
    valid = ((data["distance_parsec"] > 0) & data["distance_parsec"].notna() & data["phot_g"].notna())

    # Absolute magnitude
    data["abs_mag"] = np.nan
    data.loc[valid, "abs_mag"] = (data.loc[valid, "phot_g"] + 5 - 5 * np.log10(data.loc[valid, "distance_parsec"]))

    # Return filtered DataFrame with selected columns (fixed indexing)
    cols = ["source_id", "host_name", "teff", "spectral_type", "feh",
            "distance_parsec", "abs_mag", "bp_rp",
            "mass", "age", "age_lower", "age_upper", "ra", "dec"]
    
    return data.loc[valid, cols].reset_index(drop=True)

#print(find_charachteristics(gaia_df))

#using the function to find the characteristics of the stars in the different categories
full_char = find_charachteristics(full_stars)
print(full_char)

quasi_char = find_charachteristics(quasi_stars)
print(quasi_char)

semi_char = find_charachteristics(semi_stars)
print(semi_char)


Making an HR diagram

In [ ]:
#import colours
import matplotlib.colors as colors

#remake HR diagram but log scaled to clearly see the locations of the stars on the bigger picture
plt.figure(figsize=(8,6))
plt.style.use('seaborn-v0_8-whitegrid')
# pretty colour gradient
t_range = np.linspace(3000, 30000, len(gaia_df))

#combine all the data into one for the colour map
# all_char = pd.concat([full_char, quasi_char, semi_char])

# mask = np.where((all_char["teff"] >= 0) & (all_char["teff"] <= 30000)) 
# intermiadate_step = all_char["teff"]
vmin = min(full_char["teff"].min(), quasi_char["teff"].min(), semi_char["teff"].min())
vmax = max(full_char["teff"].max(), quasi_char["teff"].max(), semi_char["teff"].max())

# print(vmin, vmax)


sc = plt.scatter(full_char["teff"], full_char["abs_mag"], c = full_char["teff"], cmap = "coolwarm_r", s = 80, 
                 label = "Full solar-like stars", marker="o", vmin=vmin, vmax=vmax, edgecolors = "black", alpha = 0.9) 
sc1 = plt.scatter(quasi_char["teff"], quasi_char["abs_mag"], c = quasi_char["teff"], cmap = "coolwarm_r", s = 80, 
                  label = "Quasi solar-like stars", marker="^", vmin=vmin, vmax=vmax, edgecolors = "black", alpha = 0.9) 
sc2 = plt.scatter(semi_char["teff"], semi_char["abs_mag"], c = semi_char["teff"], cmap = "coolwarm_r", s = 80, 
                  label = "Semi solar-like stars", marker="s", vmin=vmin, vmax=vmax, edgecolors = "black", alpha = 0.9) 

cbar = plt.colorbar(pad=0.01, aspect=35  )
cbar.set_label("Temperature (K)", fontsize=14)
cbar.ax.tick_params(labelsize=11)

# Plotting the sun for reference
plt.scatter(5772, 4.83, marker = "*", s = 200, label = "Sun", color = "yellow", edgecolors = "black") # plotting the sun for reference

plt.xlim(3000, 30000) # limit x-axis to focus on main sequence stars
plt.ylim(-5, 16) # limit y-axis to focus on main sequence stars

plt.gca().invert_xaxis()
plt.gca().invert_yaxis()
plt.xscale('log')
plt.xlabel("Temperature (K)", fontsize = 14)
plt.ylabel("Absolute magnitude", fontsize = 14)
plt.title( "H–R Diagram: Absolute Magnitude vs. Temperature",fontsize=16,pad=15)
plt.gca().tick_params(axis='both', which='both', labelsize=14)
plt.yticks(fontsize = 14)
plt.xticks(fontsize=14)
plt.grid(True, linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig("hr.pdf")
plt.legend()
plt.show()


Inspect metallicity and temperature relation

In [ ]:
#Let us now do some statistics to try and find correlations between the different characteristics of the stars, such as temperature and metallicity.

#removes all datapoijnts where metallicity or temperature is not known, to avoid errors in the correlation test

plt.style.use('seaborn-v0_8-whitegrid')
plt.figure(figsize=(6,5), dpi=150)

full_feh_clean = full_char[full_char["feh"] != "--"]
quasi_feh_clean = quasi_char[quasi_char["feh"] != "--"]
semi_feh_clean = semi_char[semi_char["feh"] != "--"]

full_temp_clean = full_char[full_char["teff"] != "--"]
quasi_temp_clean = quasi_char[quasi_char["teff"] != "--"]
semi_temp_clean = semi_char[semi_char["teff"] != "--"]

plt.scatter(full_temp_clean["teff"], full_feh_clean["feh"], label="Full solar-like stars", marker="o", edgecolors = "black", alpha = 0.8, color='steelblue')
plt.scatter(quasi_temp_clean["teff"], quasi_feh_clean["feh"], label="Quasi solar-like stars", marker="^", edgecolors = "black", alpha = 0.8, color='hotpink')
plt.scatter(semi_temp_clean["teff"], semi_feh_clean["feh"], label="Semi solar-like stars", marker="s", edgecolors = "black", alpha = 0.8, color='forestgreen')

plt.ylabel("Metallicity [Fe/H]",  fontsize=14)
plt.xlabel("Temperature (K)", fontsize=14)
plt.title("Temperature vs Metallicity for Solar-like Stars",   fontsize=16,    pad=12)   

plt.grid(True, linestyle='--', alpha=0.3)
plt.yticks(fontsize = 14)
plt.xticks(fontsize=14)

plt.legend()
plt.tight_layout()
plt.savefig("metal.pdf")
plt.show()

from scipy.stats import kendalltau

print('Applying kendall tau test for temperature and metallicity correlation in full solar-like stars:')
print('Kendall tau for full solar-like stars:', kendalltau(full_temp_clean["teff"], full_feh_clean["feh"]))
print('Kendall tau for quasi solar-like stars:', kendalltau(quasi_temp_clean["teff"], quasi_feh_clean["feh"]))
print('Kendall tau for semi solar-like stars:', kendalltau(semi_temp_clean["teff"], semi_feh_clean["feh"]))

# combining the data for all stars to see if there is a correlation in the whole dataset
all_temp_clean = pd.concat([full_temp_clean["teff"], quasi_temp_clean["teff"], semi_temp_clean["teff"]])
all_feh_clean = pd.concat([full_feh_clean["feh"], quasi_feh_clean["feh"], semi_feh_clean["feh"]])
print('Kendall tau for all stars combined:', kendalltau(all_temp_clean, all_feh_clean))

Compare our two full solar like stars to the Sun

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

#obtaining data of only these stars and linking it to their own variables to be able to plot them in the radar chart
star_1 = full_char.iloc[0]
star_2 = full_char.iloc[1]

#normalised to sun in most cases
r_1 = [star_1["teff"]/5772, np.log10(star_1["distance_parsec"]), star_1["abs_mag"]/4.83, star_1["feh"], star_1["age"]/4.6, star_1["mass"], star_1["bp_rp"]]
r_2 = [star_2["teff"]/5772, np.log10(star_2["distance_parsec"]), star_2["abs_mag"]/4.83, star_2["feh"], star_2["age"]/4.6, star_2["mass"], star_2["bp_rp"]]
r_sun = [1, 0, 4.83/4.83, 0.0+1, np.log10(4.6), 1.0, 0.65] # Solar values for reference

categories = ['Temperature (T_sun)', 'Distance (log10(pc))', 'Absolute Magnitude (normalised to sun)', 'Metallicity', 'Age (Normalized to sun)', 'Mass (Msun)', 'Colour (bp-rp)']
fig.add_trace(go.Scatterpolar(
      r=r_1, theta=categories, fill='toself', name=star_1["host_name"]))
fig.add_trace(go.Scatterpolar(
      r=r_2, theta=categories, fill='toself', name=star_2["host_name"]))

fig.add_trace(go.Scatterpolar(r=r_sun, theta=categories, fill='toself', name='Sun', line=dict(color='lightgreen')))

Plot position in the sky compared to other stars

In [ ]:
full_cor = SkyCoord(full_char['ra'], full_char['dec'], unit=u.deg)
quasi_cor = SkyCoord(quasi_char['ra'], quasi_char['dec'], unit=u.deg)
semi_cor = SkyCoord(semi_char['ra'], semi_char['dec'], unit=u.deg)


plt.xlabel('Right Ascension (degrees)')
plt.ylabel('Declination (degrees)')
plt.title('Sky Distribution of Solar-like Stars')

# plt.subplot(111, projection="aitoff")
# plt.grid(True)

plt.plot(full_char['ra'], full_char['dec'], 'o', label='Full solar-like stars', markeredgecolor = "black", alpha = 0.7)
plt.plot(quasi_char['ra'], quasi_char['dec'], '^', label='Quasi solar-like stars', markeredgecolor = "black", alpha = 0.7)      
plt.plot(semi_char['ra'], semi_char['dec'], 's', label='Semi solar-like stars', markeredgecolor = "black", alpha = 0.7)

plt.legend()
plt.show()

In [ ]:
# importing all stars with planetary companion
from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive

#directly drop all duplicated and only use star coordinates
stars_w_c = NasaExoplanetArchive.query_criteria(
    table="pscomppars",
    select="pl_name, hostname, ra, dec, st_met, st_mass, st_teff").to_pandas().drop_duplicates(subset=["hostname"]) 

In [ ]:
#we can now include this in our plort to see if there is a difference in the distribution of stars with and without planets
full_cor = SkyCoord(full_char['ra'], full_char['dec'], unit=u.deg)
quasi_cor = SkyCoord(quasi_char['ra'], quasi_char['dec'], unit=u.deg)
semi_cor = SkyCoord(semi_char['ra'], semi_char['dec'], unit=u.deg)


plt.xlabel('Right Ascension (degrees)')
plt.ylabel('Declination (degrees)')
plt.title('Sky Distribution of Stars with confirmed exoplanets')

# plt.subplot(111, projection="aitoff")
# plt.grid(True)

plt.plot(full_char['ra'], full_char['dec'], 'o', label='Full solar-like stars', alpha = 0.7, markeredgecolor = "black")
plt.plot(quasi_char['ra'], quasi_char['dec'], '^', label='Quasi solar-like stars', alpha = 0.7, markeredgecolor = "black")      
plt.plot(semi_char['ra'], semi_char['dec'], 's', label='Semi solar-like stars', alpha = 0.7, markeredgecolor = "black")
plt.plot(stars_w_c['ra'], stars_w_c['dec'], 'x', label='Stars with exoplanets', alpha = 0.2, color = "black")

plt.legend()
plt.show()

In [ ]:
#quick analysis if the number of stars in the densely populated area is similar for all stars or only candidate stars
mask_allstars = np.where((stars_w_c['ra'] > 280) & (stars_w_c['ra'] < 320) & (stars_w_c['dec'] > 30) & (stars_w_c['dec'] < 55))
print(f'The fraction of all stars in the densely populated area is: {len(mask_allstars[0]) / len(stars_w_c)}')

candidates_combined_loc = pd.concat([full_char, quasi_char, semi_char])
mask_candiate = np.where((candidates_combined_loc['ra'] > 280) & (candidates_combined_loc['ra'] < 320) & 
                         (candidates_combined_loc['dec'] > 30) & (candidates_combined_loc['dec'] < 55))

print(f'The fraction of candidate stars in the densely populated area is: {len(mask_candiate[0]) / len(candidates_combined_loc)}')

#let us also identify how often Kepler appears in our dataset and check if it matches with the highly dense area in the sky
all_kepler = stars_w_c[stars_w_c['hostname'].str.contains("Kepler")]
solar_like_kepler = candidates_combined_loc[candidates_combined_loc['host_name'].str.contains("Kepler")]

print(f'The fraction of Kepler stars in the dataset is: {len(all_kepler)/len(stars_w_c)}')
print(f'The fraction of solar-like Kepler stars in the dataset is: {len(solar_like_kepler)/len(candidates_combined_loc)}')

Compare metallicities to other stars with exoplanets

In [ ]:
#Let us now compare the metalicities
full_feh_clean = full_char[full_char["feh"] != "--"]
quasi_feh_clean = quasi_char[quasi_char["feh"] != "--"]
semi_feh_clean = semi_char[semi_char["feh"] != "--"]

concatenate = np.concatenate([full_feh_clean["feh"], quasi_feh_clean["feh"], semi_feh_clean["feh"]])

import scipy.stats as stats

concatenate = np.array(concatenate, dtype=float)
remove_nan = ~np.isnan(stars_w_c['st_met'])
allstars_feh = stars_w_c['st_met'][remove_nan]


mu_cs, sigma_cs = stats.norm.fit(concatenate)
mu_all, sigma_all = stats.norm.fit(allstars_feh)

range_feh = np.linspace(concatenate.min(), concatenate.max(), 100)

pdf_cs = stats.norm.pdf(range_feh, loc = mu_cs, scale = sigma_cs)
prdf_all = stats.norm.pdf(range_feh, loc = mu_all, scale = sigma_all) 


fig= plt.figure(figsize=(10, 6))

#making the histogram and the fitted normal distribution for the stars with planets and all candidate stars
plt.plot(range_feh, prdf_all, label="Fitted normal distribution for stars with planets", color="black", linestyle='--')
plt.plot(range_feh,  pdf_cs, label="Fitted normal distribution", color="red", linestyle='--')
plt.hist(concatenate, bins=30, alpha=0.25, label='All candidate stars', color = "red", density=True)  
plt.hist(stars_w_c['st_met'], bins=30, alpha=0.25, label='Stars with planets', color = "black", density=True)

#annotating the mean values
plt.vlines(mu_cs, 0, max(pdf_cs), colors='red', linestyles='-', label=f'Mean candidate stars: {mu_cs:.2f} +/- {sigma_cs:.2f}')
plt.vlines(mu_all, 0, max(prdf_all), colors='black', linestyles='-', label=f'Mean stars with planets: {mu_all:.2f} +/- {sigma_all:.2f}')


#formatting the plot
plt.xlabel("Metallicity [Fe/H]")
plt.ylabel("Number of stars")
plt.title("Metallicity distribution of Solar-like Stars")
plt.legend()    
plt.show()


In [ ]:
#checking if we see a metalicity distribution difference between the different categories of stars

plt.hist(full_feh_clean['feh'], bins=30, alpha=0.5, label='Full solar-like stars', color = "blue", density=True)
plt.hist(quasi_feh_clean['feh'], bins=30, alpha=0.5, label='Quasi solar-like stars', color = "green", density=True)
plt.hist(semi_feh_clean['feh'], bins=30, alpha=0.5, label='Semi solar-like stars', color = "orange", density=True)

In [ ]:
#finding the stars with outlying metalicity to investigate why
concatenate_all = pd.concat((full_char, quasi_char, semi_char))
low_feh = np.where(concatenate_all['feh'] <=-0.5)
print(concatenate_all.iloc[low_feh[0]])

Do the same as above but for the temperatures and masses

In [ ]:
#Let us now compare the masses
full_feh_clean = full_char[full_char["mass"] != "--"]
quasi_feh_clean = quasi_char[quasi_char["mass"] != "--"]
semi_feh_clean = semi_char[semi_char["mass"] != "--"]

concatenate_mass = np.concatenate([full_feh_clean["mass"], quasi_feh_clean["mass"], semi_feh_clean["mass"]])

import scipy.stats as stats

concatenate_mass = np.array(concatenate_mass, dtype=float)
remove_nan = ~np.isnan(stars_w_c['st_mass'])
allstars_mass = stars_w_c['st_mass'][remove_nan]


mu_cs, sigma_cs = stats.norm.fit(concatenate_mass)
mu_all, sigma_all = stats.norm.fit(allstars_mass)

range_mass = np.linspace(allstars_mass.min(), allstars_mass.max(), 1000)

pdf_cs = stats.norm.pdf(range_mass, loc = mu_cs, scale = sigma_cs)
prdf_all = stats.norm.pdf(range_mass, loc = mu_all, scale = sigma_all) 


fig= plt.figure(figsize=(10, 6))

#making the histogram and the fitted normal distribution for the stars with planets and all candidate stars
plt.plot(range_mass, prdf_all, label="Fitted normal distribution for stars with planets", color="black", linestyle='--')
plt.plot(range_mass,  pdf_cs, label="Fitted normal distribution", color="red", linestyle='--')
plt.hist(concatenate_mass, bins=20, alpha=0.25, label='All candidate stars', color = "red", density=True)  
plt.hist(allstars_mass, bins=20, alpha=0.25, label='Stars with planets', color = "black", density=True)

#annotating the mean values
plt.vlines(mu_cs, 0, max(pdf_cs), colors='red', linestyles='-', label=f'Mean candidate stars: {mu_cs:.2f}')
plt.vlines(mu_all, 0, max(prdf_all), colors='black', linestyles='-', label=f'Mean stars with planets: {mu_all:.2f}')

print(len(concatenate_mass))

plt.xlim(0, 3)
#formatting the plot
plt.xlabel("Mass (solar masses)")
plt.ylabel("Number of stars")
plt.title("Mass distribution of Solar-like Stars")
plt.legend()    
plt.show()

In [ ]:
#Let us now compare the temperatures
full_tef_clean = full_char[full_char["teff"] != "--"]
quasi_tef_clean = quasi_char[quasi_char["teff"] != "--"]
semi_tef_clean = semi_char[semi_char["teff"] != "--"]

concatenate_tef = np.concatenate([full_tef_clean["teff"], quasi_tef_clean["teff"], semi_tef_clean["teff"]])

import scipy.stats as stats

concatenate_tef = np.array(concatenate_tef, dtype=float)
remove_nan = ~np.isnan(stars_w_c['st_teff'])
allstars_tef = stars_w_c['st_teff'][remove_nan]


mu_cs, sigma_cs = stats.norm.fit(concatenate_tef)
mu_all, sigma_all = stats.norm.fit(allstars_tef)

range_tef = np.linspace(allstars_tef.min(), allstars_tef.max(), 1000)

pdf_cs = stats.norm.pdf(range_tef, loc = mu_cs, scale = sigma_cs)
prdf_all = stats.norm.pdf(range_tef, loc = mu_all, scale = sigma_all) 


fig= plt.figure(figsize=(10, 6))

#making the histogram and the fitted normal distribution for the stars with planets and all candidate stars
plt.plot(range_tef, prdf_all, label="Fitted normal distribution for stars with planets", color="black", linestyle='--')
plt.plot(range_tef,  pdf_cs, label="Fitted normal distribution", color="red", linestyle='--')
plt.hist(concatenate_tef, bins=50, alpha=0.25, label='All candidate stars', color = "red", density=True)  
plt.hist(allstars_tef, bins=50, alpha=0.25, label='Stars with planets', color = "black", density=True)

#annotating the mean values
plt.vlines(mu_cs, 0, max(pdf_cs), colors='red', linestyles='-', label=f'Mean candidate stars: {mu_cs:.2f}')
plt.vlines(mu_all, 0, max(prdf_all), colors='black', linestyles='-', label=f'Mean stars with planets: {mu_all:.2f}')

#formatting the plot
plt.xlabel("Temperature (K)")
plt.xlim(0, 10000)
plt.ylabel("Number of stars")
plt.title("Temperature distribution of Solar-like Stars")
plt.legend()    
plt.show()

In [ ]:
#let us research if the candidate stars disitribution can be drawn from the all the stars with planets 

In [ ]:
# Let us compare the metallicity to absolute magnitude to see if there is a correlation between the two
plt.scatter(full_char["feh"], full_char["abs_mag"], label="Full solar-like stars", marker="o", edgecolors = "black", alpha = 0.7)
plt.scatter(quasi_char["feh"], quasi_char["abs_mag"], label="Quasi solar-like stars", marker="^", edgecolors = "black", alpha = 0.7)
plt.scatter(semi_char["feh"], semi_char["abs_mag"], label="Semi solar-like stars", marker="s", edgecolors = "black", alpha = 0.7)  


plt.legend()
plt.xlabel("Metallicity [Fe/H]")
plt.ylabel("Absolute magnitude")